# Text2SQL-LoRA: fine-tune Qwen2.5-0.5B with LoRA on a free Colab T4

Runtime → Change runtime type → **T4 GPU**, then Run all (about 25 minutes).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/khush-ahuja/Text2SQL-LoRA
%cd Text2SQL-LoRA
!pip -q install -U peft datasets accelerate

In [ ]:
!python train.py --model Qwen/Qwen2.5-0.5B-Instruct --n-train 20000 --epochs 1 --batch 16 --out runs/qwen05b-lora

In [ ]:
!python evaluate.py --adapter runs/qwen05b-lora/adapter --n-test 500

In [ ]:
import json
print(json.dumps(json.load(open('runs/qwen05b-lora/eval.json')), indent=2))
s = json.load(open('runs/qwen05b-lora/train_summary.json'))
print({k: v for k, v in s.items() if k != 'loss_curve'})

In [ ]:
# a few examples where the fine-tuned model fixed the base model's answer
rows = [json.loads(l) for l in open('runs/qwen05b-lora/eval_samples.jsonl')]
base = {r['question']: r for r in rows if r['model'] == 'base'}
fixed = [r for r in rows if r['model'] == 'lora' and r['correct'] and base[r['question']]['correct'] is False]
for r in fixed[:5]:
    print('Q:', r['question'])
    print('base:', base[r['question']]['pred'])
    print('lora:', r['pred'], '\n')

In [ ]:
!python infer.py --adapter runs/qwen05b-lora/adapter \
  --schema "CREATE TABLE employees (name VARCHAR, dept VARCHAR, salary INTEGER)" \
  --question "Which department has the highest average salary?"

In [ ]:
# optional: download the adapter (~35 MB) and results
!cd runs && zip -qr qwen05b-lora.zip qwen05b-lora
from google.colab import files
files.download('runs/qwen05b-lora.zip')